# Stage 10 — Demand Forecast: Tiered Model Selection
**Dashboard page:** Demand Forecast
**Tabs:** Stage 10 Forecast · Fused Demand (M2)

**Tiers:** 0=Zero · 1=HistAvg · 2=Croston · 3=AutoETS vs LightGBM · 4=AutoETS vs LightGBM vs NHITS
**Selection:** 1-window cross-validation holdout MAE per tier-eligible SKU

In [ ]:
import sys, warnings, re
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INTERIM   = PROJECT_ROOT / "data" / "interim"
PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS   = PROJECT_ROOT / "data" / "outputs"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
})
PALETTE = ["#4361EE","#EF4444","#2CC56F","#F59E0B","#A855F7",
           "#64748B","#06B6D4","#F97316","#10B981","#8B5CF6"]
STATUS_COLORS = {"stockout":"#EF4444","critical":"#F97316",
                 "low":"#F59E0B","ok":"#2CC56F","excess":"#4361EE"}
TIER_COLORS   = {"critical":"#EF4444","managed":"#F97316",
                 "watch":"#F59E0B","rationalise":"#94A3B8"}

def load(name, base=None):
    if base:
        p = Path(base) / name
        if p.exists(): return pd.read_parquet(p)
    for b in [INTERIM, PROCESSED, OUTPUTS]:
        p = b / name
        if p.exists(): return pd.read_parquet(p)
    raise FileNotFoundError(f"{name} not found")

def fmt_lkr(v):
    if abs(v) >= 1e9: return f"LKR {v/1e9:.1f}B"
    if abs(v) >= 1e6: return f"LKR {v/1e6:.1f}M"
    return f"LKR {v:,.0f}"


In [ ]:
dfc  = load("demand_forecast.parquet")
mv   = load("monthly_demand.parquet")

print(f"Demand forecast : {len(dfc):,} SKUs")
print(f"Columns         : {dfc.columns.tolist()}")
print()
print("Method distribution:")
print(dfc["method"].value_counts().to_string())


## Tab 1 — Stage 10: Method Breakdown

In [ ]:
mc = dfc["method"].value_counts()
fig,axes = plt.subplots(1,2,figsize=(13,5))
mc.plot(kind="bar",ax=axes[0],color=PALETTE[:len(mc)],edgecolor="white")
axes[0].set_title("Forecast Method Assigned per SKU
(Tier 0→4 based on active_months + XYZ class)")
axes[0].set_ylabel("SKUs"); axes[0].tick_params(axis="x",rotation=30)
for bar,val in zip(axes[0].patches,mc.values):
    axes[0].text(bar.get_x()+bar.get_width()/2,bar.get_height()+5,str(val),ha="center",fontsize=9)
axes[1].pie(mc.values,labels=mc.index,colors=PALETTE[:len(mc)],autopct="%1.1f%%",startangle=90)
axes[1].set_title("Method Share"); plt.tight_layout(); plt.show()

# Forecast qty distribution by method
fc_col = "forecast_lt" if "forecast_lt" in dfc.columns else "forecast_m1"
fig,ax = plt.subplots(figsize=(11,5))
methods = dfc["method"].dropna().unique()
boxes = [dfc.loc[dfc["method"]==m,fc_col].clip(upper=dfc[fc_col].quantile(0.95)).dropna().tolist() for m in methods]
bp = ax.boxplot(boxes,labels=[str(m) for m in methods],patch_artist=True,showfliers=False)
for i,patch in enumerate(bp["boxes"]):
    patch.set_facecolor(PALETTE[i%len(PALETTE)]); patch.set_alpha(0.6)
ax.set_title("3-Month Lead-Time Forecast Distribution by Method")
ax.set_ylabel("Forecast qty"); ax.tick_params(axis="x",rotation=30)
plt.tight_layout(); plt.show()


## Historical Monthly Demand

In [ ]:
month_col = "year_month_str"; issue_col = "issue_qty"; ret_col = "return_qty"
if month_col in mv.columns and issue_col in mv.columns:
    agg = mv.groupby(month_col).agg(issues=(issue_col,"sum"),returns=(ret_col,"sum")).sort_index()
    agg["net"] = agg["issues"] - agg["returns"]
    fig,ax = plt.subplots(figsize=(13,4))
    agg["net"].plot(ax=ax,color=PALETTE[0],lw=2)
    agg["issues"].plot(ax=ax,color=PALETTE[0],lw=1,ls=":",alpha=0.5,label="Gross issues")
    ax.set_title("Historical Monthly Net Demand — All SKUs
(issues - returns)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"{int(x):,}"))
    ax.tick_params(axis="x",rotation=45); ax.legend(); plt.tight_layout(); plt.show()


## Tab 2 — Fused Demand (M2: orders + sales + UIO blend)

In [ ]:
try:
    fused = load("m2_fused_demand.parquet")
    print(f"Fused demand: {len(fused):,} rows | cols: {fused.columns.tolist()}")
    print(fused.head(5).to_string())

    dc_col   = next((c for c in ["demand_class","demand_category","method"] if c in fused.columns),None)
    qty_col  = next((c for c in ["fused_demand","demand","forecast_lt"] if c in fused.columns),None)
    mat_col  = "material_9" if "material_9" in fused.columns else None

    if dc_col and qty_col:
        fig,axes = plt.subplots(1,2,figsize=(13,5))
        dc = fused[dc_col].value_counts()
        dc.plot(kind="bar",ax=axes[0],color=PALETTE[:len(dc)],edgecolor="white")
        axes[0].set_title("Fused Demand: Source/Class Distribution
(blends orders+sales+UIO signals)")
        axes[0].set_ylabel("SKUs"); axes[0].tick_params(axis="x",rotation=30)

        d = fused[qty_col].clip(upper=fused[qty_col].quantile(0.95))
        axes[1].hist(d.dropna(),bins=60,color=PALETTE[4],edgecolor="white",alpha=0.8)
        axes[1].set_title("Fused Lead-Time Demand Distribution")
        axes[1].set_xlabel("Units (3-month)"); plt.tight_layout(); plt.show()
except FileNotFoundError:
    print("M2 fused demand not yet computed. Run Module 2 from the Pipeline page.")
